# Japan BESS Research Lab v2

独立复算资产，保护 v1 Notebook。A=官方数据，B=计算，C=假设。2026 YTD 样本外回测不是未来盈利预测；Historical Resampling Scenario 不是未来概率分布。

完整离线再生：`prepare_lab_v2.py → settle_lab_strategies.mjs → finalize_lab_v2.py`。本 Notebook 从正式标准化数据复算关键样本、泄漏扰动、结算、风险与规则，不在每次打开时重跑 12,510 次 LP。Python NumPy/SciPy 与 Node 必须可用；任何官方数据更新应重新冻结基线。


In [1]:
from pathlib import Path
import sys, json, hashlib, subprocess
import numpy as np
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'scripts/lab_core.py').exists())
sys.path.insert(0,str(ROOT/'scripts'))
from lab_core import load_prices, anatomy, feature_day, fit_ridge, predict_ridge, same_slot_baseline, historical_resampling, reservation_dispatch, frontier_value
dates, matrices = load_prices()
lab = json.loads((ROOT/'dist/data/lab-v2.json').read_text(encoding='utf-8'))
assert len(dates)==643 and matrices['chubu'].shape==(643,48)
sha=hashlib.sha256((ROOT/'research/jepx-normalized.csv').read_bytes()).hexdigest()
assert sha==lab['lineage']['source_sha256']
print({'days':len(dates),'slots':len(dates)*48,'timezone':'Asia/Tokyo','unit':'JPY/kWh','sha256':sha})


{'days': 643, 'slots': 30864, 'timezone': 'Asia/Tokyo', 'unit': 'JPY/kWh', 'sha256': '58cb77975c3a0dd26dd432072c24063a3a1a450615c34d891d58738949e77b90'}


## Market Anatomy：五区域同槽分位数与日内价差
均按 JST 受渡日，不用本机时区。2025 同期比较为 1/1–10/5。

In [2]:
for region,prices in matrices.items():
    a=anatomy(prices,dates,list(range(365,643)))
    expected=lab['anatomy'][region]['2026ytd']
    np.testing.assert_allclose(a['slot_median'],expected['slot_median'])
    np.testing.assert_allclose(a['monthly'][0]['slot_mean'],expected['monthly'][0]['slot_mean'])
    print(region,{'spread_p10_p50_p90':a['spread_quantiles'],'negative_slots':a['negative_slots'],'high_slot_share':a['events']['high']['share']})


chubu {'spread_p10_p50_p90': [7.413000000000001, 15.325, 26.131], 'negative_slots': 0, 'high_slot_share': 0.07531474820143885}
tokyo {'spread_p10_p50_p90': [6.404000000000001, 14.59, 33.754999999999995], 'negative_slots': 0, 'high_slot_share': 0.07336630695443645}
kansai {'spread_p10_p50_p90': [6.747, 14.285, 25.105000000000004], 'negative_slots': 0, 'high_slot_share': 0.043914868105515585}
kyushu {'spread_p10_p50_p90': [6.629, 14.44, 24.891000000000002], 'negative_slots': 0, 'high_slot_share': 0.027577937649880094}
tohoku {'spread_p10_p50_p90': [7.769, 18.025, 30.082], 'negative_slots': 0, 'high_slot_share': 0.03162470023980815}


## 防未来泄漏：改动目标日及之后真实价格
决策 D−1 08:00 JST。D−1 受渡价格在 D−2 拍卖产生；历史档案没有发布修订版本，信息可得时点是 C 假设。每月扩展训练，不用目标月未来标签。

In [3]:
target='2026-06-17';i=dates.index(target);cutoff=dates.index('2026-06-01')-1
x=matrices['chubu'];changed=x.copy();changed[i:]=changed[i:]*100+9999
before=fit_ridge(x,dates,cutoff);after=fit_ridge(changed,dates,cutoff)
np.testing.assert_array_equal(feature_day(x,dates,i),feature_day(changed,dates,i))
np.testing.assert_array_equal(same_slot_baseline(x,i),same_slot_baseline(changed,i))
np.testing.assert_array_equal(predict_ridge(before,x,dates,i),predict_ridge(after,changed,dates,i))
print({'target':target,'training_end':dates[cutoff],'future_perturbation_equal':True,'ridge_alpha':100,'standardization':'training-only'})


{'target': '2026-06-17', 'training_end': '2026-05-31', 'future_perturbation_equal': True, 'ridge_alpha': 100, 'standardization': 'training-only'}


## 预测计划与实际结算分离，48槽回放
保持 v1 2MW/8.14MWh、88%效率、SOC首尾10%、80格、每日最多1次EFC；毛收益不含交易费或固定成本。

In [4]:
node_code = """
import {readFileSync} from 'node:fs';
import {planAndSettle} from './dist/lab-model.js';
const j=p=>JSON.parse(readFileSync(p,'utf8')),date='2026-06-17';
const actual=j('dist/data/prices.json').days.find(x=>x.date===date).chubu;
const forecast=j('research/lab-v2/forecasts.json').chubu.find(x=>x.date===date).ridge;
const r=planAndSettle(actual,forecast);
console.log(JSON.stringify({date,slots:r.rows.length,grossRevenue:r.grossRevenue,chargingCost:r.chargingCost,dischargeRevenue:r.dischargeRevenue,finalSoc:r.rows.at(-1).socAfter,cumulative:r.rows.at(-1).cumulativeRevenue,equivalentCycles:r.equivalentCycles}));
"""
replay=json.loads(subprocess.check_output(['node','--input-type=module','-e',node_code],cwd=ROOT,text=True))
expected=next(r for r in lab['series']['chubu']['ridge'] if r['date']==replay['date'])
assert abs(expected['grossRevenue']-replay['grossRevenue'])<1e-6
assert replay['slots']==48 and abs(replay['finalSoc']-.1)<1e-8
assert abs(replay['cumulative']-replay['grossRevenue'])<1e-6
print(replay)
print({k:{f:v[f] for f in ['grossRevenue','captureRatio','mae','lossDays']} for k,v in lab['metrics']['chubu'].items()})


{'date': '2026-06-17', 'slots': 48, 'grossRevenue': 47741.1894949862, 'chargingCost': 128773.27745093912, 'dischargeRevenue': 176514.46694592532, 'finalSoc': 0.1, 'cumulative': 47741.18949498623, 'equivalentCycles': 0.8}
{'perfect': {'grossRevenue': 21010852.593611002, 'captureRatio': 1.0, 'mae': None, 'lossDays': 0}, 'median28': {'grossRevenue': 14311813.23758111, 'captureRatio': 0.6811628977842173, 'mae': 4.121863384292566, 'lossDays': 22}, 'ridge': {'grossRevenue': 15016727.05949025, 'captureRatio': 0.7147128843337157, 'mae': 3.044901279130544, 'lossDays': 18}}


## Historical Resampling Scenario
每月分层、循环7日块、2000次、seed20261007；保留相同历史区间长度，不是未来收益置信区间。

In [5]:
daily=lab['series']['chubu']['ridge']
scenario=historical_resampling([x['date'] for x in daily],[x['grossRevenue'] for x in daily],seed=20261007)
assert scenario==lab['risk']['chubu']['ridge']['resampling']
assert scenario['p10_p50_p90']==sorted(scenario['p10_p50_p90'])
print(scenario)


{'type': 'Historical Resampling Scenario', 'method': 'month-stratified circular 7-day moving blocks', 'seed': 20261007, 'replicates': 2000, 'block_days': 7, 'days': 278, 'p10_p50_p90': [14048105.300942555, 14996857.449423209, 15943520.55229645], 'interpretation': 'same historical interval composition; not a future confidence interval or probability forecast'}


## 资源预留与规则粒度：对应连续模型基线
Frontier全天扣留功率，付费小时仅影响容量额；首尾SOC50%，静态5min双向headroom。未模拟激活、恢复、成交概率或处罚。粒度实验是已知价 Flexibility Upper Bound。cap15→10只是容量天花板，不是clearing price。

In [6]:
price=matrices['chubu'][dates.index('2026-01-19')]
zero=reservation_dispatch(price,0);reserve=reservation_dispatch(price,1)
coarse=reservation_dispatch(price,1,180);fine=reservation_dispatch(price,1,30)
assert fine['grossRevenue']>=coarse['grossRevenue']-1e-4
assert sum(coarse['reserveMask'])==sum(fine['reserveMask'])==24
value=frontier_value(reserve['grossRevenue'],zero['grossRevenue'],1,1,3,12,.65,.06)
assert abs(frontier_value(reserve['grossRevenue'],zero['grossRevenue'],1,1,value['breakEvenPrice'],12,.65,.06)['combinedScenario']-zero['grossRevenue'])<1e-6
print({'0MW':zero['grossRevenue'],'1MW':reserve['grossRevenue'],'3h':coarse['grossRevenue'],'30min':fine['grossRevenue'],'flexibility_upper_bound':fine['grossRevenue']-coarse['grossRevenue']})
print(value)
print({'old_cap15_theoretical_daily':1000*12*2*.65*15,'new_cap10_theoretical_daily':1000*12*2*.65*10,'not_clearing_price':True})


{'0MW': 70465.3111445265, '1MW': 52855.77768226981, '3h': 69275.03836156867, '30min': 70188.25003202903, 'flexibility_upper_bound': 913.2116704603541}
{'reserveMW': 1, 'spotPowerMW': 1, 'spotGross': 52855.77768226981, 'lostSpotValue': 17609.533462256688, 'capacityGross': 46800.0, 'capacityFee': 936.0, 'capacityAfterFee': 45864.0, 'combinedScenario': 98719.7776822698, 'breakEvenPrice': 1.188816247580557, 'contractedUnits': 15600.0}
{'old_cap15_theoretical_daily': 234000.0, 'new_cap10_theoretical_daily': 156000.0, 'not_clearing_price': True}
